# BRSET download via AWS (PhysioNet, version 1.0.2), revision 2

Downloads BRSET through PhysioNet's AWS access point, verifies every file against PhysioNet's SHA256SUMS, and stores it in Drive at `stochastic_orthogonality/data/brset/`. **It never opens or summarises the labels.** It records only file counts, sizes, checksums and the time in `brset_access_record.json`, so it can be shown that the data were not examined before the BRSET analysis plan was written.

Needs Colab secrets `AWS_ACCESS_KEY_ID` and `AWS_SECRET_ACCESS_KEY` (the `physionet-download` IAM user linked to PhysioNet), with notebook access on, and about 17 GB free in Drive. CPU runtime.

Data use: credentialed PhysioNet data. Keep it in your own Drive and Colab only; do not share files, record-level outputs or images with anyone, including AI services.

Revision 2 replaces the direct PhysioNet download (too slow, about 2 to 3 images a minute) with the AWS access point. If an earlier attempt left files in this runtime, they are reused when they match.

In [1]:
# Cell 1: setup
from google.colab import drive, userdata
drive.mount('/content/drive')
!pip -q install awscli
import os, glob, subprocess, hashlib, json, datetime, tarfile, shutil
from concurrent.futures import ThreadPoolExecutor
os.environ['AWS_ACCESS_KEY_ID'] = userdata.get('AWS_ACCESS_KEY_ID')
os.environ['AWS_SECRET_ACCESS_KEY'] = userdata.get('AWS_SECRET_ACCESS_KEY')
os.environ['AWS_DEFAULT_REGION'] = 'us-east-1'
VERSION = '1.0.2'
AP = 's3://arn:aws:s3:us-east-1:724665945834:accesspoint/brazilian-ophthalmological-v1-0-2-01/brazilian-ophthalmological/1.0.2/'
SRC = f'/content/brset_dl/physionet.org/files/brazilian-ophthalmological/{VERSION}'; os.makedirs(SRC, exist_ok=True)
DEST = '/content/drive/MyDrive/stochastic_orthogonality/data/brset'; os.makedirs(DEST, exist_ok=True)
ident = subprocess.run(['aws', 'sts', 'get-caller-identity'], capture_output=True, text=True)
print('AWS identity check:', 'ok' if ident.returncode == 0 and 'physionet-download' in ident.stdout else ident.stderr[-300:])
print('free local disk (GB):', round(shutil.disk_usage('/content').free / 1e9, 1))

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.2/20.2 MB 70.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.5/570.5 kB 30.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sphinx 8.2.3 requires docutils<0.22,>=0.20, but you have docutils 0.19 which is incompatible.
AWS identity check: ok
free local disk (GB): 220.3


In [2]:
# Cell 2: download (resumable; rerun if interrupted). Progress in the Terminal:
#   du -sm /content/brset_dl; find /content/brset_dl -name '*.jpg' | wc -l
subprocess.run(['aws', 'configure', 'set', 'default.s3.max_concurrent_requests', '32'])
for p in glob.glob(f'{SRC}/**/*.part', recursive=True): os.remove(p)
r = subprocess.run(['aws', 's3', 'sync', AP, SRC, '--only-show-errors'], capture_output=True, text=True)
print('exit code:', r.returncode); print(r.stderr[-1500:])
if 'AccessDenied' in r.stderr or 'Access Denied' in r.stderr: print('Access denied: PhysioNet may still be registering the AWS link. Wait ten minutes and rerun this cell.')
n = sum(len(f) for _, _, f in os.walk(SRC)); size = sum(os.path.getsize(os.path.join(d, x)) for d, _, f in os.walk(SRC) for x in f)
print(f'{n} files, {size / 1e9:.2f} GB')

exit code: 0

16269 files, 16.71 GB


In [3]:
# Cell 3: verify every file against SHA256SUMS (prints counts only)
def sha(p, chunk=1 << 22):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for b in iter(lambda: f.read(chunk), b''): h.update(b)
    return h.hexdigest()
expected = {}
for line in open(f'{SRC}/SHA256SUMS.txt'):
    parts = line.strip().split(maxsplit=1)
    if len(parts) == 2: expected[parts[1].lstrip('*').lstrip('./')] = parts[0]
def check(item):
    rel, h = item; p = f'{SRC}/{rel}'
    return rel, (os.path.exists(p) and sha(p) == h)
with ThreadPoolExecutor(8) as ex: res = list(ex.map(check, expected.items()))
bad = [rel for rel, ok in res if not ok]
print(f'{len(expected)} files listed; {len(expected) - len(bad)} verified; {len(bad)} missing or mismatched')
if bad:
    for rel in bad:
        if os.path.exists(f'{SRC}/{rel}'): os.remove(f'{SRC}/{rel}')
    raise SystemExit('Removed the bad files. Rerun Cell 2 (it fetches only what is missing), then this cell.')

16268 files listed; 16268 verified; 0 missing or mismatched


In [4]:
# Cell 4: store in Drive as one archive plus the small files; write the access record
listed = set(expected) | {'SHA256SUMS.txt'}
tar_local = f'/content/brset_{VERSION}.tar'
if os.path.exists(tar_local): os.remove(tar_local)
with tarfile.open(tar_local, 'w') as t:
    for rel in sorted(listed): t.add(f'{SRC}/{rel}', arcname=f'brset_{VERSION}/{rel}')
for f in ('label_brset.csv', 'SHA256SUMS.txt', 'LICENSE.txt'):
    if os.path.exists(f'{SRC}/{f}'): shutil.copy(f'{SRC}/{f}', f'{DEST}/{f}')
shutil.copy(tar_local, f'{DEST}/brset_{VERSION}.tar')
images = [r for r in expected if r.lower().endswith(('.jpg', '.jpeg', '.png'))]
record = {'downloaded_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(), 'source': AP, 'route': 'PhysioNet AWS access point', 'version': VERSION,
          'files_verified_against_SHA256SUMS': len(expected), 'image_files': len(images),
          'archive': f'brset_{VERSION}.tar', 'archive_sha256': sha(tar_local), 'archive_bytes': os.path.getsize(tar_local),
          'label_file_sha256': sha(f'{SRC}/label_brset.csv'), 'labels_examined': False,
          'note': 'Downloaded and verified only. label_brset.csv was not opened or summarised. The BRSET analysis plan is to be written before any label is read.'}
json.dump(record, open(f'{DEST}/brset_access_record.json', 'w'), indent=2)
drive.flush_and_unmount(); drive.mount('/content/drive')
print(json.dumps({k: v for k, v in record.items() if k != 'note'}, indent=1)); print('Stored in', DEST)

Mounted at /content/drive
{
 "downloaded_utc": "2026-10-03T01:46:17.278519+00:00",
 "source": "s3://arn:aws:s3:us-east-1:724665945834:accesspoint/brazilian-ophthalmological-v1-0-2-01/brazilian-ophthalmological/1.0.2/",
 "route": "PhysioNet AWS access point",
 "version": "1.0.2",
 "files_verified_against_SHA256SUMS": 16268,
 "image_files": 16266,
 "archive": "brset_1.0.2.tar",
 "archive_sha256": "151bf6a71b51ba1a773203d506e0d8b67785b1058667a666f1826fb57f530aaa",
 "archive_bytes": 16740874240,
 "label_file_sha256": "4cd000ec1f651deb5cfb4f018ed1dec7a874da59a20c47f1e59a421d4a112adf",
 "labels_examined": false
}
Stored in /content/drive/MyDrive/stochastic_orthogonality/data/brset
